# Capa bronce

Todas las capas comparten `data/processed/database.duckdb` y se separan por esquemas: `bronce`, `plata` y `oro`. Este notebook escribe solo en `bronce`, donde todas las columnas son `VARCHAR`, incluidas las de trazabilidad. Los campos vacíos se conservan como `NULL`; los valores literales como `NA` se conservan como texto.

## MITMA · Viajes diarios por distrito

Entradas: `.csv.gz` diarios. El rango de fechas es inclusivo; `None` deja ese extremo abierto.

In [9]:
import csv
import gzip
from datetime import UTC, date, datetime
from pathlib import Path

import duckdb

def find_project_root():
    current_dir = Path.cwd()
    for candidate in (current_dir, *current_dir.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Open this notebook from within the project directory.')


PROJECT_ROOT = find_project_root()
DUCKDB_PATH = PROJECT_ROOT / 'data/processed/database.duckdb'
TRIPS_DIR = PROJECT_ROOT / 'data/mitma/por-distritos/viajes/diarios'

TRIPS_START_DATE = None
TRIPS_END_DATE = None

### Seleccionar y comprobar los archivos

La fecha de selección se toma del nombre del archivo. Comprobamos las cabeceras antes de escribir en DuckDB para detectar cambios de columnas o de orden.

In [10]:
TRIP_COLUMNS = [
    'fecha', 'periodo', 'origen', 'destino', 'distancia',
    'actividad_origen', 'actividad_destino',
    'estudio_origen_posible', 'estudio_destino_posible',
    'residencia', 'renta', 'edad', 'sexo', 'viajes', 'viajes_km',
]

if TRIPS_START_DATE and TRIPS_END_DATE and TRIPS_START_DATE > TRIPS_END_DATE:
    raise ValueError('TRIPS_START_DATE must not be later than TRIPS_END_DATE.')

trip_files = []
for path in sorted(TRIPS_DIR.glob('????????_Viajes_distritos.csv.gz')):
    file_date = date.fromisoformat(path.name[:8])
    if TRIPS_START_DATE and file_date < TRIPS_START_DATE:
        continue
    if TRIPS_END_DATE and file_date > TRIPS_END_DATE:
        continue
    trip_files.append(path)

if not trip_files:
    raise FileNotFoundError(f'No trip files found in {TRIPS_DIR} for the selected date range.')

print(f'Selected {len(trip_files)} files:')
for path in trip_files:
    with gzip.open(path, 'rt', encoding='utf-8-sig', newline='') as source:
        header = next(csv.reader(source, delimiter='|'), [])
    if header != TRIP_COLUMNS:
        raise ValueError(f'Unexpected header in {path.name}: {header}')
    print(path.relative_to(PROJECT_ROOT))

Selected 5 files:
data/mitma/por-distritos/viajes/diarios/20230101_Viajes_distritos.csv.gz
data/mitma/por-distritos/viajes/diarios/20230102_Viajes_distritos.csv.gz
data/mitma/por-distritos/viajes/diarios/20230103_Viajes_distritos.csv.gz
data/mitma/por-distritos/viajes/diarios/20230104_Viajes_distritos.csv.gz
data/mitma/por-distritos/viajes/diarios/20230105_Viajes_distritos.csv.gz


### Cargar en DuckDB

`_archivo` guarda la ruta relativa al proyecto y `_timestamp` guarda la hora UTC del lote en formato ISO 8601.

Cada ejecución sustituye solo las filas de los archivos seleccionados. Si falla la lectura de alguno, la transacción deshace todo el lote.

In [11]:
trip_column_types = dict.fromkeys(TRIP_COLUMNS, 'VARCHAR')
trip_column_definitions = ', '.join(f'"{column}" VARCHAR' for column in TRIP_COLUMNS)
batch_timestamp = datetime.now(UTC).isoformat()

DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('BEGIN TRANSACTION')
    try:
        con.execute('CREATE SCHEMA IF NOT EXISTS bronce')
        con.execute(f'''
            CREATE TABLE IF NOT EXISTS bronce.mitma_viajes (
                {trip_column_definitions},
                _archivo VARCHAR NOT NULL,
                _timestamp VARCHAR NOT NULL
            )
        ''')

        for path in trip_files:
            relative_path = path.relative_to(PROJECT_ROOT).as_posix()
            print(f'Loading {relative_path}')
            con.execute('DELETE FROM bronce.mitma_viajes WHERE _archivo = ?', [relative_path])
            con.execute('''
                INSERT INTO bronce.mitma_viajes BY NAME
                SELECT *, ? AS _archivo, ? AS _timestamp
                FROM read_csv(
                    ?, delim = '|', header = true, auto_detect = false,
                    columns = ?, nullstr = '', strict_mode = true
                )
            ''', [relative_path, batch_timestamp, str(path), trip_column_types])

        con.execute('COMMIT')
    except BaseException:
        con.execute('ROLLBACK')
        raise

print(f'Batch committed: {batch_timestamp}')

Loading data/mitma/por-distritos/viajes/diarios/20230101_Viajes_distritos.csv.gz
Loading data/mitma/por-distritos/viajes/diarios/20230102_Viajes_distritos.csv.gz
Loading data/mitma/por-distritos/viajes/diarios/20230103_Viajes_distritos.csv.gz
Loading data/mitma/por-distritos/viajes/diarios/20230104_Viajes_distritos.csv.gz
Loading data/mitma/por-distritos/viajes/diarios/20230105_Viajes_distritos.csv.gz
Batch committed: 2026-11-19T13:01:39.190727+00:00


### Inspeccionar la tabla

Al recargar la misma selección, los recuentos deben mantenerse y cada archivo recargado debe aparecer solo con el nuevo `_timestamp`.

In [12]:
from IPython.display import display

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    schema = con.sql('DESCRIBE bronce.mitma_viajes').df()
    assert schema['column_type'].eq('VARCHAR').all(), 'All columns must be VARCHAR'
    display(schema[['column_name', 'column_type']])
    display(con.sql('SELECT * FROM bronce.mitma_viajes LIMIT 10').df())
    display(con.sql('''
        SELECT _archivo, _timestamp, count(*) AS row_count
        FROM bronce.mitma_viajes
        GROUP BY _archivo, _timestamp
        ORDER BY _archivo, _timestamp
    ''').df())
    display(con.sql('''
        SUMMARIZE bronce.mitma_viajes
    ''').df())

,column_name,column_type
0,fecha,VARCHAR
1,periodo,VARCHAR
2,origen,VARCHAR
3,destino,VARCHAR
4,distancia,VARCHAR
5,actividad_origen,VARCHAR
6,actividad_destino,VARCHAR
7,estudio_origen_posible,VARCHAR
8,estudio_destino_posible,VARCHAR
9,residencia,VARCHAR


,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km,_archivo,_timestamp
0,20230101,00,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,12.806,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
1,20230101,00,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,5.202,22.098,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
2,20230101,01,01001,01001,2-10,no_frecuente,no_frecuente,no,no,31,>15,25-45,NA,4.648,20.805,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
3,20230101,02,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.28,9.213,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
4,20230101,03,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,2.922,11.72,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
5,20230101,03,01001,01001,2-10,trabajo_estudio,casa,no,no,01,10-15,NA,NA,7.219,28.876,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
6,20230101,07,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,11.72,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
7,20230101,07,01001,01001,2-10,casa,trabajo_estudio,no,no,01,10-15,NA,NA,2.881,10.91,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
8,20230101,08,01001,01001,2-10,frecuente,frecuente,no,no,01,>15,45-65,mujer,8.183,32.712,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00
9,20230101,09,01001,01001,2-10,casa,trabajo_estudio,no,no,01,10-15,NA,NA,3.134,13.079,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00


,_archivo,_timestamp,row_count
0,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00,14642142
1,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00,16988306
2,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00,18528055
3,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00,18890961
4,data/mitma/por-distritos/viajes/diarios/202301...,2026-11-19T13:01:39.190727+00:00,19696534


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230105,5,None,None,None,None,None,88745998,0.0
1,periodo,VARCHAR,00,23,24,None,None,None,None,None,88745998,0.0
2,origen,VARCHAR,01001,externo,3902,None,None,None,None,None,88745998,0.0
3,destino,VARCHAR,01001,externo,3902,None,None,None,None,None,88745998,0.0
4,distancia,VARCHAR,0.5-2,>50,4,None,None,None,None,None,88745998,0.0
5,actividad_origen,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,88745998,0.0
6,actividad_destino,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,88745998,0.0
7,estudio_origen_posible,VARCHAR,no,si,2,None,None,None,None,None,88745998,0.0
8,estudio_destino_posible,VARCHAR,no,si,2,None,None,None,None,None,88745998,0.0
9,residencia,VARCHAR,01,52,53,None,None,None,None,None,88745998,0.0


## MITMA · Pernoctaciones diarias por distrito

El rango de fechas es inclusivo e independiente del de viajes. Al recargar, solo se reemplazan las filas de los archivos seleccionados.

In [13]:
OVERNIGHTS_DIR = PROJECT_ROOT / 'data/mitma/por-distritos/pernoctaciones/diarios'
OVERNIGHTS_START_DATE = None
OVERNIGHTS_END_DATE = None
OVERNIGHT_COLUMNS = ['fecha', 'zona_residencia', 'zona_pernoctacion', 'personas']

if OVERNIGHTS_START_DATE and OVERNIGHTS_END_DATE and OVERNIGHTS_START_DATE > OVERNIGHTS_END_DATE:
    raise ValueError('OVERNIGHTS_START_DATE must not be later than OVERNIGHTS_END_DATE.')

overnight_files = []
for path in sorted(OVERNIGHTS_DIR.glob('????????_Pernoctaciones_distritos.csv.gz')):
    file_date = date.fromisoformat(path.name[:8])
    if OVERNIGHTS_START_DATE and file_date < OVERNIGHTS_START_DATE:
        continue
    if OVERNIGHTS_END_DATE and file_date > OVERNIGHTS_END_DATE:
        continue
    overnight_files.append(path)

if not overnight_files:
    raise FileNotFoundError(f'No overnight files found in {OVERNIGHTS_DIR} for the selected date range.')

print(f'Selected {len(overnight_files)} files:')
for path in overnight_files:
    with gzip.open(path, 'rt', encoding='utf-8-sig', newline='') as source:
        header = next(csv.reader(source, delimiter='|'), [])
    if header != OVERNIGHT_COLUMNS:
        raise ValueError(f'Unexpected header in {path.name}: {header}')
    print(path.relative_to(PROJECT_ROOT))

Selected 5 files:
data/mitma/por-distritos/pernoctaciones/diarios/20230101_Pernoctaciones_distritos.csv.gz
data/mitma/por-distritos/pernoctaciones/diarios/20230102_Pernoctaciones_distritos.csv.gz
data/mitma/por-distritos/pernoctaciones/diarios/20230103_Pernoctaciones_distritos.csv.gz
data/mitma/por-distritos/pernoctaciones/diarios/20230104_Pernoctaciones_distritos.csv.gz
data/mitma/por-distritos/pernoctaciones/diarios/20230105_Pernoctaciones_distritos.csv.gz


In [14]:
overnight_column_types = dict.fromkeys(OVERNIGHT_COLUMNS, 'VARCHAR')
overnight_batch_timestamp = datetime.now(UTC).isoformat()

DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronce')
    con.execute('''
        CREATE TABLE IF NOT EXISTS bronce.mitma_pernoctaciones (
            fecha VARCHAR,
            zona_residencia VARCHAR,
            zona_pernoctacion VARCHAR,
            personas VARCHAR,
            _archivo VARCHAR NOT NULL,
            _timestamp VARCHAR NOT NULL
        )
    ''')

    for path in overnight_files:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path}')
        con.execute('DELETE FROM bronce.mitma_pernoctaciones WHERE _archivo = ?', [relative_path])
        con.execute('''
            INSERT INTO bronce.mitma_pernoctaciones BY NAME
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = '|', header = true, auto_detect = false,
                columns = ?, nullstr = '', strict_mode = true
            )
        ''', [relative_path, overnight_batch_timestamp, str(path), overnight_column_types])

    con.commit()

print(f'Batch committed: {overnight_batch_timestamp}')

Loading data/mitma/por-distritos/pernoctaciones/diarios/20230101_Pernoctaciones_distritos.csv.gz
Loading data/mitma/por-distritos/pernoctaciones/diarios/20230102_Pernoctaciones_distritos.csv.gz
Loading data/mitma/por-distritos/pernoctaciones/diarios/20230103_Pernoctaciones_distritos.csv.gz
Loading data/mitma/por-distritos/pernoctaciones/diarios/20230104_Pernoctaciones_distritos.csv.gz
Loading data/mitma/por-distritos/pernoctaciones/diarios/20230105_Pernoctaciones_distritos.csv.gz
Batch committed: 2026-11-19T13:02:21.301036+00:00


In [15]:
from IPython.display import display

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    schema = con.sql('DESCRIBE bronce.mitma_pernoctaciones').df()
    assert schema['column_type'].eq('VARCHAR').all(), 'All columns must be VARCHAR'
    display(schema[['column_name', 'column_type']])
    display(con.sql('SELECT * FROM bronce.mitma_pernoctaciones LIMIT 10').df())
    display(con.sql('''
        SELECT _archivo, _timestamp, count(*) AS row_count
        FROM bronce.mitma_pernoctaciones
        GROUP BY _archivo, _timestamp
        ORDER BY _archivo, _timestamp
    ''').df())

,column_name,column_type
0,fecha,VARCHAR
1,zona_residencia,VARCHAR
2,zona_pernoctacion,VARCHAR
3,personas,VARCHAR
4,_archivo,VARCHAR
5,_timestamp,VARCHAR


,fecha,zona_residencia,zona_pernoctacion,personas,_archivo,_timestamp
0,20230101,01001,01001,2329.644,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
1,20230101,01001,01017_AM,16.682,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
2,20230101,01001,01028_AM,2.922,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
3,20230101,01001,01051,6.056,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
4,20230101,01001,01054_AM,5.897,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
5,20230101,01001,01058_AM,2.975,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
6,20230101,01001,0105901,21.716,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
7,20230101,01001,0105902,8.336,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
8,20230101,01001,0105903,86.286,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00
9,20230101,01001,0105904,24.691,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00


,_archivo,_timestamp,row_count
0,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00,726907
1,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00,597227
2,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00,530905
3,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00,514605
4,data/mitma/por-distritos/pernoctaciones/diario...,2026-11-19T13:02:21.301036+00:00,487895


## MITMA · Zonificación por distrito

Se cargan los límites y centroides de distritos y la tabla de relación INE–MITMA. Los nombres de distritos, municipios y GAU se reúnen en `bronce.mitma_nombres_zonas`: `tipo_zona` distingue los niveles porque un mismo `ID` puede aparecer en varios. No se cargan los CSV de población ni las geometrías de municipios o GAU.

La extensión `spatial` de DuckDB lee los shapefiles. La primera instalación puede requerir conexión. En bronce, la geometría se guarda como WKT (`geom_wkt`) con su sistema de referencia original (`crs`), sin reproyectar y manteniendo todas las columnas como `VARCHAR`.

In [16]:
ZONING_DIR = PROJECT_ROOT / 'data/mitma/zonificación'
DISTRICTS_DIR = ZONING_DIR / 'zonificacion_distritos'
ZONING_SOURCES = [
    ('bronce.mitma_zonificacion_distritos', DISTRICTS_DIR / 'zonificacion_distritos.shp'),
    ('bronce.mitma_centroides_distritos', DISTRICTS_DIR / 'zonificacion_distritos_centroides.shp'),
    ('bronce.mitma_relacion_ine', ZONING_DIR / 'relacion_ine_zonificacionMitma.csv'),
]
ZONE_NAMES_TABLE = 'bronce.mitma_nombres_zonas'
ZONE_NAME_SOURCES = [
    ('distrito', DISTRICTS_DIR / 'nombres_distritos.csv'),
    ('municipio', ZONING_DIR / 'zonificacion_municipios/nombres_municipios.csv'),
    ('gau', ZONING_DIR / 'zonificacion_GAU/nombres_gaus.csv'),
]

for _, path in ZONING_SOURCES + ZONE_NAME_SOURCES:
    if not path.is_file():
        raise FileNotFoundError(path)

zoning_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('INSTALL spatial')
    con.execute('LOAD spatial')
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronce')

    for table, path in ZONING_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        if path.suffix == '.shp':
            source_sql = '''
                SELECT ID, ST_AsText(geom) AS geom_wkt, ST_CRS(geom) AS crs
                FROM ST_Read(?)
            '''
        else:
            source_sql = '''
                SELECT * FROM read_csv(
                    ?, delim = '|', header = true, all_varchar = true,
                    nullstr = '', strict_mode = true
                )
            '''
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM ({source_sql}) AS source
        ''', [relative_path, zoning_batch_timestamp, str(path)])

    con.execute(f'''
        CREATE OR REPLACE TABLE {ZONE_NAMES_TABLE} (
            ID VARCHAR,
            name VARCHAR,
            tipo_zona VARCHAR NOT NULL,
            _archivo VARCHAR NOT NULL,
            _timestamp VARCHAR NOT NULL
        )
    ''')
    for zone_type, path in ZONE_NAME_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {ZONE_NAMES_TABLE}')
        con.execute(f'''
            INSERT INTO {ZONE_NAMES_TABLE} BY NAME
            SELECT ID, name, ? AS tipo_zona, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = '|', header = true, all_varchar = true,
                nullstr = '', strict_mode = true
            )
        ''', [zone_type, relative_path, zoning_batch_timestamp, str(path)])

    con.commit()

print(f'Batch committed: {zoning_batch_timestamp}')

Loading data/mitma/zonificación/zonificacion_distritos/zonificacion_distritos.shp into bronce.mitma_zonificacion_distritos
Loading data/mitma/zonificación/zonificacion_distritos/zonificacion_distritos_centroides.shp into bronce.mitma_centroides_distritos
Loading data/mitma/zonificación/relacion_ine_zonificacionMitma.csv into bronce.mitma_relacion_ine
Loading data/mitma/zonificación/zonificacion_distritos/nombres_distritos.csv into bronce.mitma_nombres_zonas
Loading data/mitma/zonificación/zonificacion_municipios/nombres_municipios.csv into bronce.mitma_nombres_zonas
Loading data/mitma/zonificación/zonificacion_GAU/nombres_gaus.csv into bronce.mitma_nombres_zonas
Batch committed: 2026-11-19T13:02:23.108723+00:00


In [17]:
from IPython.display import display

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    tables = [table for table, _ in ZONING_SOURCES] + [ZONE_NAMES_TABLE]
    for table in tables:
        print(table)
        schema = con.sql(f'DESCRIBE {table}').df()
        assert schema['column_type'].eq('VARCHAR').all(), 'All columns must be VARCHAR'
        display(con.sql(f'''
            SELECT _archivo, _timestamp, count(*) AS row_count
            FROM {table}
            GROUP BY _archivo, _timestamp
        ''').df())
        if 'geom_wkt' in schema['column_name'].values:
            display(con.sql(f'SELECT ID, crs, left(geom_wkt, 80) AS wkt_preview FROM {table} LIMIT 5').df())
        else:
            display(con.sql(f'SELECT * EXCLUDE (_archivo, _timestamp) FROM {table} LIMIT 5').df())

bronce.mitma_zonificacion_distritos


,_archivo,_timestamp,row_count
0,data/mitma/zonificación/zonificacion_distritos...,2026-11-19T13:02:23.108723+00:00,3909


,ID,crs,wkt_preview
0,01001,EPSG:3042,MULTIPOLYGON (((538090.1842999998 4737372.7314...
1,01002,EPSG:3042,MULTIPOLYGON (((501984.88879999984 4753913.869...
2,01004_AM,EPSG:3042,MULTIPOLYGON (((488613.9693999998 4778775.7973...
3,01009_AM,EPSG:3042,"POLYGON ((554658.5522849594 4740793.545559007,..."
4,01010,EPSG:3042,"POLYGON ((499938.4358000001 4768365.7598, 4993..."


bronce.mitma_centroides_distritos


,_archivo,_timestamp,row_count
0,data/mitma/zonificación/zonificacion_distritos...,2026-11-19T13:02:23.108723+00:00,3909


,ID,crs,wkt_preview
0,01001,EPSG:3042,POINT (539945.6938472011 4741948.88264596)
1,01002,EPSG:3042,POINT (502306.66342558316 4763642.870954381)
2,01004_AM,EPSG:3042,POINT (493797.445728896 4777478.589138445)
3,01009_AM,EPSG:3042,POINT (546467.0607110746 4747997.339160551)
4,01010,EPSG:3042,POINT (493632.64651308063 4769207.650992133)


... (2 outputs left out)

,seccion_ine,distrito_ine,municipio_ine,distrito_mitma,municipio_mitma,gau_mitma
0,0100101001,0100101,01001,01001,01001,01001
1,0100101002,0100101,01001,01001,01001,01001
2,0100201001,0100201,01002,01002,01002,01002
3,0100201002,0100201,01002,01002,01002,01002
4,0100201003,0100201,01002,01002,01002,01002


bronce.mitma_nombres_zonas


,_archivo,_timestamp,row_count
0,data/mitma/zonificación/zonificacion_distritos...,2026-11-19T13:02:23.108723+00:00,3792
1,data/mitma/zonificación/zonificacion_GAU/nombr...,2026-11-19T13:02:23.108723+00:00,2086
2,data/mitma/zonificación/zonificacion_municipio...,2026-11-19T13:02:23.108723+00:00,2618


,ID,name,tipo_zona
0,01001,Alegría-Dulantzi,distrito
1,01002,Amurrio,distrito
2,01004_AM,Artziniega agregacion de municipios,distrito
3,01009_AM,Asparrena agregacion de municipios,distrito
4,01010,Ayala/Aiara,distrito


## INE · Población, nivel de estudios y relación con la actividad

`poblacion.csv` usa punto y coma; `nivel_estudios.csv` y `relacion_actividad.csv` usan tabuladores. Los archivos locales de población y nivel de estudios terminan con una fila incompleta: se conserva y las columnas finales que faltan quedan en `NULL`.

In [18]:
INE_DIR = PROJECT_ROOT / 'data/ine'
INE_SOURCES = [
    ('bronce.ine_poblacion', INE_DIR / 'poblacion.csv', ';'),
    ('bronce.ine_nivel_estudios', INE_DIR / 'nivel_estudios.csv', '\t'),
    ('bronce.ine_relacion_actividad', INE_DIR / 'relacion_actividad.csv', '\t'),
]

for _, path, _ in INE_SOURCES:
    if not path.is_file():
        raise FileNotFoundError(path)

ine_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronce')
    for table, path, delimiter in INE_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = ?, header = true, all_varchar = true,
                nullstr = '', null_padding = true, strict_mode = true
            )
        ''', [relative_path, ine_batch_timestamp, str(path), delimiter])
    con.commit()

print(f'Batch committed: {ine_batch_timestamp}')

Loading data/ine/poblacion.csv into bronce.ine_poblacion
Loading data/ine/nivel_estudios.csv into bronce.ine_nivel_estudios
Loading data/ine/relacion_actividad.csv into bronce.ine_relacion_actividad
Batch committed: 2026-11-19T13:02:24.368960+00:00


In [19]:
from IPython.display import display

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    for table, _, _ in INE_SOURCES:
        print(table)
        schema = con.sql(f'DESCRIBE {table}').df()
        assert schema['column_type'].eq('VARCHAR').all(), 'All columns must be VARCHAR'
        display(con.sql(f'''
            SELECT _archivo, _timestamp, count(*) AS row_count
            FROM {table}
            GROUP BY _archivo, _timestamp
        ''').df())
        display(con.sql(f'SELECT * FROM {table} LIMIT 5').df())

bronce.ine_poblacion


,_archivo,_timestamp,row_count
0,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00,34842


,Total Nacional,Provincias,Municipios,Secciones,Sexo,Lugar de nacimiento,Periodo,Total,_archivo,_timestamp
0,Total Nacional,None,None,None,Total,Total,2025,49.128.297,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00
1,Total Nacional,None,None,None,Total,Total,2024,48.619.695,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00
2,Total Nacional,None,None,None,Total,Total,2023,48.085.361,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00
3,Total Nacional,None,None,None,Total,Total,2022,47.486.727,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00
4,Total Nacional,None,None,None,Total,Total,2021,47.400.798,data/ine/poblacion.csv,2026-11-19T13:02:24.368960+00:00


bronce.ine_nivel_estudios


,_archivo,_timestamp,row_count
0,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00,598820


,Total Nacional,Provincias,Municipios,Secciones,País de nacimiento,Nivel de formación alcanzado,Periodo,Total,_archivo,_timestamp
0,Total Nacional,None,None,None,Total,Total,2024,42.189.318,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00
1,Total Nacional,None,None,None,Total,Total,2023,41.548.146,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00
2,Total Nacional,None,None,None,Total,Total,2022,40.899.207,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00
3,Total Nacional,None,None,None,Total,Total,2021,40.686.130,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00
4,Total Nacional,None,None,None,Total,Educación primaria e inferior,2024,6.237.407,data/ine/nivel_estudios.csv,2026-11-19T13:02:24.368960+00:00


bronce.ine_relacion_actividad


,_archivo,_timestamp,row_count
0,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00,3234240


,Total Nacional,Provincias,Municipios,Secciones,País de nacimiento,Relación con la actividad,Periodo,Total,_archivo,_timestamp
0,Total Nacional,None,None,None,Total,Total,2024,41.645.105,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00
1,Total Nacional,None,None,None,Total,Total,2023,41.030.806,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00
2,Total Nacional,None,None,None,Total,Total,2022,40.391.547,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00
3,Total Nacional,None,None,None,Total,Total,2021,40.191.793,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00
4,Total Nacional,None,None,None,Total,Ocupado/a,2024,20.373.148,data/ine/relacion_actividad.csv,2026-11-19T13:02:24.368960+00:00


## INE · Catálogo de municipios y relación provincia–comunidad autónoma

`diccionario26.xlsx` contiene el catálogo de municipios a 1 de enero de 2026. `provincias_ccaa.csv` es una transcripción de la [tabla oficial del INE](https://www.ine.es/daco/daco42/codmun/cod_ccaa_provincia.htm), incluidas Ceuta y Melilla. Las dos tablas se cargan por separado, sin geometría ni cruces. Para identificar un municipio, se concatena `CPRO || CMUN` (sin `DC`); para obtener los nombres de provincia y comunidad o ciudad autónoma, se cruza por `CPRO`.

In [20]:
INE_CATALOG_DIR = PROJECT_ROOT / 'data/ine'
INE_CATALOG_SOURCES = [
    ('bronce.ine_municipios', INE_CATALOG_DIR / 'diccionario26.xlsx', '''
        read_xlsx(?, range = 'A2:E1048576', header = true,
                  all_varchar = true, stop_at_empty = true)
    '''),
    ('bronce.ine_provincias_ccaa', INE_CATALOG_DIR / 'provincias_ccaa.csv', '''
        read_csv(?, delim = ';', header = true, all_varchar = true,
                 nullstr = '', strict_mode = true)
    '''),
]

for _, path, _ in INE_CATALOG_SOURCES:
    if not path.is_file():
        raise FileNotFoundError(path)

catalog_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('INSTALL excel')
    con.execute('LOAD excel')
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronce')
    for table, path, reader in INE_CATALOG_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM {reader}
        ''', [relative_path, catalog_batch_timestamp, str(path)])
    con.commit()

print(f'Batch committed: {catalog_batch_timestamp}')

Loading data/ine/diccionario26.xlsx into bronce.ine_municipios
Loading data/ine/provincias_ccaa.csv into bronce.ine_provincias_ccaa
Batch committed: 2026-11-19T13:02:25.932269+00:00


In [21]:
from IPython.display import display

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    for table, _, _ in INE_CATALOG_SOURCES:
        print(table)
        schema = con.sql(f'DESCRIBE {table}').df()
        assert schema['column_type'].eq('VARCHAR').all(), 'All columns must be VARCHAR'
        display(schema[['column_name', 'column_type']])
        display(con.sql(f'''
            SELECT _archivo, _timestamp, count(*) AS row_count
            FROM {table}
            GROUP BY _archivo, _timestamp
        ''').df())
        display(con.sql(f'SELECT * FROM {table} LIMIT 5').df())

bronce.ine_municipios


,column_name,column_type
0,CODAUTO,VARCHAR
1,CPRO,VARCHAR
2,CMUN,VARCHAR
3,DC,VARCHAR
4,NOMBRE,VARCHAR
5,_archivo,VARCHAR
6,_timestamp,VARCHAR


,_archivo,_timestamp,row_count
0,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00,8132


,CODAUTO,CPRO,CMUN,DC,NOMBRE,_archivo,_timestamp
0,16,01,051,3,Agurain/Salvatierra,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00
1,16,01,001,4,Alegría-Dulantzi,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00
2,16,01,002,9,Amurrio,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00
3,16,01,049,3,Añana,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00
4,16,01,003,5,Aramaio,data/ine/diccionario26.xlsx,2026-11-19T13:02:25.932269+00:00


bronce.ine_provincias_ccaa


,column_name,column_type
0,CODAUTO,VARCHAR
1,COMUNIDAD_AUTONOMA,VARCHAR
2,CPRO,VARCHAR
3,PROVINCIA,VARCHAR
4,_archivo,VARCHAR
5,_timestamp,VARCHAR


,_archivo,_timestamp,row_count
0,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00,52


,CODAUTO,COMUNIDAD_AUTONOMA,CPRO,PROVINCIA,_archivo,_timestamp
0,01,Andalucía,04,Almería,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00
1,01,Andalucía,11,Cádiz,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00
2,01,Andalucía,14,Córdoba,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00
3,01,Andalucía,18,Granada,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00
4,01,Andalucía,21,Huelva,data/ine/provincias_ccaa.csv,2026-11-19T13:02:25.932269+00:00
